# Tissue-Guided Bias Field Correction Examples

This notebook demonstrates two complementary tissue-guided bias field correction workflows:

1. **GMM-based correction** for higher-frequency residual bias fields, constrained by tissue segmentation labels.
2. **RBF scatter correction** for lower-frequency residual bias fields, using adaptive radial basis function control points.

The examples use the sample T1-weighted image and tissue label map in `resources/`. Outputs are written to `examples/outputs/`.

## Setup

Run this notebook from the project root or from the `examples/` directory. The setup cell resolves paths automatically.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'resources').exists() else cwd.parent
resources_dir = project_root / 'resources'
outputs_dir = project_root / 'examples' / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)

input_img = resources_dir / 'T1w_04mm.nii.gz'
input_lab = resources_dir / 'T1w_label_04mm.nii.gz'

assert input_img.exists(), f'Missing input image: {input_img}'
assert input_lab.exists(), f'Missing input label: {input_lab}'

print(f'Project root: {project_root}')
print(f'Input image:  {input_img}')
print(f'Input label:  {input_lab}')
print(f'Outputs:      {outputs_dir}')

In [ ]:
import SimpleITK as sitk

image = sitk.ReadImage(str(input_img))
label = sitk.ReadImage(str(input_lab))

print('Image size:', image.GetSize())
print('Image spacing:', image.GetSpacing())
print('Label size:', label.GetSize())
print('Label spacing:', label.GetSpacing())

## Example 1: GMM-based high-frequency correction

Use the Gaussian mixture strategy when residual bias varies at a relatively high spatial frequency. The method estimates bias while using tissue labels to constrain class intensities.

In this example, the T1-weighted image is also used as the positive-valued processing region mask, matching the common command-line pattern for this script.

In [ ]:
gmm_output_img = outputs_dir / 'T1w_bfc_gmm_04mm.nii.gz'
gmm_output_bias = outputs_dir / 'T1w_bfc_gmm_bias_04mm.nii.gz'

gmm_cmd = [
    sys.executable, str(project_root / 'bfc_gauss_mixture.py'),
    '--input_img', str(input_img),
    '--input_lab', str(input_lab),
    '--input_msk', str(input_img),
    '--output_img', str(gmm_output_img),
    '--output_bias', str(gmm_output_bias),
    '--bias_max_iter', '20',
    '--max_kernel', '7.0',
    '--min_kernel', '3.0',
    '--refine_max_iter', '1',
    '--label_soft_kernel', '0.25',
    '--modality', 'T1',
]

print(' '.join(gmm_cmd))
subprocess.run(gmm_cmd, check=True)

## Example 2: RBF scatter low-frequency correction

Use the RBF scatter strategy when residual bias is smooth and low-frequency. The method fits adaptive radial basis function control points inside the tissue-guided mask.

This example uses hard tissue labels by passing `--no_soft_labels`, matching the provided usage pattern. The label map is used as both the tissue guide and processing mask.

In [ ]:
rbf_output_img = outputs_dir / 'T1w_bfc_rbf_04mm.nii.gz'
rbf_output_bias = outputs_dir / 'T1w_bfc_rbf_bias_04mm.nii.gz'

rbf_cmd = [
    sys.executable, str(project_root / 'bfc_rbf_scatter.py'),
    '--input_img', str(input_img),
    '--input_lab', str(input_lab),
    '--input_msk', str(input_lab),
    '--output_img', str(rbf_output_img),
    '--output_bias', str(rbf_output_bias),
    '--no_soft_labels',
]

print(' '.join(rbf_cmd))
subprocess.run(rbf_cmd, check=True)

## Quick output check

The following cell verifies that each workflow wrote a corrected image and an estimated bias field.

In [ ]:
for path in [gmm_output_img, gmm_output_bias, rbf_output_img, rbf_output_bias]:
    if not path.exists():
        raise FileNotFoundError(path)
    img = sitk.ReadImage(str(path))
    print(f'{path.name}: size={img.GetSize()}, spacing={img.GetSpacing()}')

## Method selection summary

- Choose **GMM-based correction** when the remaining bias field has finer spatial variation and tissue-constrained intensity modeling is needed.
- Choose **RBF scatter correction** when the remaining bias field is broad and smoothly varying across the image.
- Inspect the saved bias fields to confirm that the estimated correction scale matches the observed residual intensity inhomogeneity.